# Prepare the dataset

Load and inspect the hand-curated examples.

In [4]:
import json
from pathlib import Path

import ipywidgets as widgets
from IPython.display import clear_output, display

DATASET_DIR = Path('dataset')

In [5]:
examples = {path.stem: json.loads(path.read_text()) for path in sorted(DATASET_DIR.glob('*.json'))}

In [6]:
SYSTEM_COLOR = '34'
USER_COLOR = '34'
ASSISTANT_COLOR = '32'
FIELD_COLOR = '38;5;245'

selector = widgets.Dropdown(
    options=list(examples),
    description='Example:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='720px'),
)
transcript = widgets.Output(
    layout=widgets.Layout(
        width='720px',
        min_height='340px',
        border='1px solid',
        padding='12px',
        overflow='auto',
    )
)


def render(name):
    items = examples[name]['input']

    with transcript:
        clear_output(wait=True)
        for item in items:
            if item['type'] == 'message':
                role = 'system' if item['role'] == 'developer' else item['role']
                content = item['content']
                text = content if isinstance(content, str) else ''.join(part['text'] for part in content)
                color = {'system': SYSTEM_COLOR, 'user': USER_COLOR, 'assistant': ASSISTANT_COLOR}[role]
                print(f'\033[{color}m[{role}]\033[0m\n{text}\n')
            elif item['type'] == 'function_call':
                print(f"\033[{ASSISTANT_COLOR}m[assistant -> {item['name']}()]\033[0m")
                arguments = json.loads(item['arguments'])
                for name, value in arguments.items():
                    print(f"\033[{FIELD_COLOR}m{name}:\033[0m {value}")
                print()
            elif item['type'] == 'function_call_output':
                print(f'\033[{ASSISTANT_COLOR}m[tool result]\033[0m')
                result = json.loads(item['output'])
                for name, value in result.items():
                    if name == 'output':
                        print(f"\033[{FIELD_COLOR}m{name}:\033[0m\n{value}")
                    else:
                        print(f"\033[{FIELD_COLOR}m{name}:\033[0m {value}")
                print()


selector.observe(lambda change: render(change['new']), names='value')
render(selector.value)
display(widgets.VBox([selector, transcript], layout=widgets.Layout(gap='12px')))